<a href="https://colab.research.google.com/github/hotusiki/Cheminformatics_pipeline/blob/main/MD_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universal MD Colab pipeline — protein + nucleic acid ligand **or** small-molecule ligand

This notebook is a **carefully generalized** version of the original RNase H + RNA:DNA (m6A) workflow.

It keeps the same overall logic:

1. prepare inputs,
2. build AMBER topology in LEaP,
3. run equilibration and production MD in OpenMM,
4. analyze the trajectory,
5. optionally run MM/GBSA.




In [ ]:
#@title **Install Conda Colab**
#@markdown It will restart the kernel (session), don't worry.
# !pip install -q condacolab
# import condacolab
# condacolab.install()
!pip install -q condacolab
import condacolab
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")

In [ ]:
#@title Install dependencies (Colab-safe)
import subprocess, re, os, textwrap

def run(cmd):
    r = subprocess.run(["bash","-lc", cmd], text=True, capture_output=True)
    return r.returncode, r.stdout, r.stderr

def must(cmd):
    code, out, err = run(cmd)
    if code != 0:
        print(out)
        print(err)
        raise RuntimeError(f"Command failed: {cmd}")
    return out.strip()

print("Python:", must("python -V"))
print("Conda :", must("conda --version"))

# Настройки conda (без строгих пинов, чтобы solver меньше страдал)
must("conda config --add channels conda-forge || true")
must("conda config --set channel_priority flexible")
must("conda config --set solver libmamba || true")

# Чистим возможные lock-и после прошлых падений
must("rm -rf /usr/local/pkgs/cache/*.lock /usr/local/pkgs/.locks || true")
must("rm -rf /usr/local/conda/pkgs/cache/*.lock /usr/local/conda/pkgs/.locks || true")

pkgs = "ambertools openmm parmed mdtraj mdanalysis pdbfixer ipywidgets"
base_cmd = f"conda install -y -c conda-forge {pkgs}"

print("Installing:", pkgs)

# 1) первая попытка
code, out, err = run(base_cmd)
if code != 0 and ("ClobberError" in err or "path collision" in err):
    # вытаскиваем конфликтующие пути
    paths = re.findall(r"path collision for '([^']+)'", err)
    paths = sorted(set(paths))
    print("\n[WARN] ClobberError detected. Removing colliding paths and retrying...")
    for p in paths:
        # удаляем только то, что conda ругает
        run(f"rm -f /usr/local/{p} || true")
    code, out, err = run(base_cmd)

# 2) если всё ещё не получилось — последняя попытка с --clobber
if code != 0:
    print(out)
    print(err)
    print("\n[WARN] Retry with --clobber (safe in Colab runtime)...")
    must(base_cmd + " --clobber")

print("\nOK: conda install finished.")

# Проверка импортов
import parmed as pmd
import openmm
import mdtraj as md
import MDAnalysis as mda

print("ParmEd     :", pmd.__version__)
print("OpenMM     :", openmm.__version__)
print("mdtraj     :", md.__version__)
print("MDAnalysis :", mda.__version__)
print("\nAll good.")

In [ ]:
!conda install -y -c conda-forge ambertools openmm parmed mdtraj mdanalysis pdbfixer -vv

In [ ]:
# Helper: run cpptraj (works even if pytraj is unavailable)
import subprocess, os
from pathlib import Path

def run_cpptraj(prmtop, trajin, commands, out_log):
    """Run cpptraj with given commands (list of strings)."""
    inp = "\n".join([f"parm {prmtop}", f"trajin {trajin}"] + commands + ["run", "quit"]) + "\n"
    inp_path = Path(out_log).with_suffix(".in")
    inp_path.write_text(inp)
    r = subprocess.run(["bash","-lc", f"cpptraj -i {inp_path} > {out_log} 2>&1"])
    if r.returncode != 0:
        tail = Path(out_log).read_text(errors="replace").splitlines()[-120:]
        print("\n".join(tail))
        raise RuntimeError(f"cpptraj failed, see {out_log}")
    return str(out_log)

print("cpptraj:", subprocess.run(["bash","-lc","which cpptraj"], text=True, capture_output=True).stdout.strip())

In [ ]:
#@title 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2) Global settings + universal switches ✅
from pathlib import Path
import os, json, hashlib, re, shutil, subprocess

#@markdown ## Paths and main mode
WORKDIR_STR = "/content/drive/MyDrive/oncotarget_pipeline/md/a2ar" #@param {type:"string"}
PROTEIN_INPUT_NAME = "A2AR.pdb" #@param {type:"string"}
LIGAND_KIND = "small_molecule" #@param ["nucleic_acid", "small_molecule"]
LIGAND_INPUT_NAME = "adrix.sdf" #@param {type:"string"}
SYSTEM_BASENAME = "a2ar_adrix" #@param {type:"string"}
JOB_PREFIX = "prot_lig" #@param {type:"string"}

#@markdown ## Nucleic-acid ligand options
MODIFIED_NA_RESNAME = "6MA" #@param {type:"string"}
MODIFIED_MARKER_ATOMS = "C01,N01" #@param {type:"string"}

#@markdown ## Small-molecule ligand options
SMALL_MOL_RESNAME = "LIG" #@param {type:"string"}
SMALL_MOL_NET_CHARGE = 0 #@param {type:"integer"}
SMALL_MOL_CHARGE_METHOD = "bcc" #@param ["bcc", "gas"]
GAFF_VERSION = "gaff2" #@param ["gaff2", "gaff"]

#@markdown ## Metals / solvent / ions
KEEP_METALS = True #@param {type:"boolean"}
METAL_ELEMENTS = "MN,MG,ZN,CA,FE,CU,CO,NI" #@param {type:"string"}
METAL_TO_ANALYZE = "auto" #@param {type:"string"}
SOLVATE_SYSTEM = True #@param {type:"boolean"}
WATER_PADDING_ANG = 8.0 #@param {type:"number"}
NEUTRALIZE_SYSTEM = True #@param {type:"boolean"}
EXTRA_NA = 0 #@param {type:"integer"}
EXTRA_CL = 0 #@param {type:"integer"}

#@markdown ## MD parameters
TEMPERATURE_K = 310.15 #@param {type:"number"}
PRESSURE_BAR  = 1.0 #@param {type:"number"}
DT_FS = 2.0 #@param {type:"number"}
FRICTION_PS = 1.0 #@param {type:"number"}

EQUIL_STEPS_TOTAL = 1000000 #@param {type:"integer"}
EQUIL_DCD_PS = 100.0 #@param {type:"number"}
EQUIL_LOG_PS = 10.0 #@param {type:"number"}
EQUIL_CHK_PS = 100.0 #@param {type:"number"}

PROD_STEPS_TO_RUN = 5000000 #@param {type:"integer"}
PROD_DCD_PS = 5.0 #@param {type:"number"}
PROD_LOG_PS = 5.0 #@param {type:"number"}
PROD_CHK_PS = 50.0 #@param {type:"number"}

WORKDIR = Path(WORKDIR_STR)
WORKDIR.mkdir(parents=True, exist_ok=True)

PROT_IN = WORKDIR / PROTEIN_INPUT_NAME
LIG_IN  = WORKDIR / LIGAND_INPUT_NAME

MARKER_ATOMS = {x.strip().upper() for x in MODIFIED_MARKER_ATOMS.split(",") if x.strip()}
METAL_ELEMENTS_LIST = [x.strip().upper() for x in METAL_ELEMENTS.split(",") if x.strip()]

LIGAND_CLEAN_PDB = WORKDIR / f"{SYSTEM_BASENAME}_ligand_clean.pdb"
LIGAND_LEAP_PDB  = WORKDIR / f"{SYSTEM_BASENAME}_ligand_LEaP_view.pdb"
LIGAND_MOL2      = WORKDIR / f"{SYSTEM_BASENAME}_ligand_{GAFF_VERSION}.mol2"
LIGAND_FRCMOD    = WORKDIR / f"{SYSTEM_BASENAME}_ligand_{GAFF_VERSION}.frcmod"
LIGAND_LIB       = WORKDIR / f"{SYSTEM_BASENAME}_ligand_{GAFF_VERSION}.lib"

PROT_AMBER_PDB = WORKDIR / f"{Path(PROTEIN_INPUT_NAME).stem}_amber.pdb"
RAW_COMPLEX_PDB = WORKDIR / f"{SYSTEM_BASENAME}_raw_complex.pdb"
PREPARED_COMPLEX_PDB = WORKDIR / f"{SYSTEM_BASENAME}_prepared_complex.pdb"

SYSTEM_PRMTOP = WORKDIR / f"SYS_{SYSTEM_BASENAME}.prmtop"
SYSTEM_CRD    = WORKDIR / f"SYS_{SYSTEM_BASENAME}.crd"
SYSTEM_PDB    = WORKDIR / f"SYS_{SYSTEM_BASENAME}.pdb"

EQUIL_JOB = f"{JOB_PREFIX}_equil"
PROD_JOB  = f"{JOB_PREFIX}_prod"

CONFIG_JSON = WORKDIR / f"{SYSTEM_BASENAME}_run_config.json"

def sh(cmd, cwd=None, check=False):
    print("$", cmd)
    r = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True
    )
    if r.stdout:
        print(r.stdout[-3000:])
    if r.stderr:
        print(r.stderr[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed with code {r.returncode}: {cmd}")
    return r

def md5_head(path: Path, nbytes=2_000_000):
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read(nbytes))
    return h.hexdigest()

config = {
    "WORKDIR": str(WORKDIR),
    "PROTEIN_INPUT_NAME": PROTEIN_INPUT_NAME,
    "LIGAND_KIND": LIGAND_KIND,
    "LIGAND_INPUT_NAME": LIGAND_INPUT_NAME,
    "SYSTEM_BASENAME": SYSTEM_BASENAME,
    "JOB_PREFIX": JOB_PREFIX,
    "MODIFIED_NA_RESNAME": MODIFIED_NA_RESNAME,
    "MARKER_ATOMS": sorted(MARKER_ATOMS),
    "SMALL_MOL_RESNAME": SMALL_MOL_RESNAME,
    "SMALL_MOL_NET_CHARGE": int(SMALL_MOL_NET_CHARGE),
    "SMALL_MOL_CHARGE_METHOD": SMALL_MOL_CHARGE_METHOD,
    "GAFF_VERSION": GAFF_VERSION,
    "KEEP_METALS": bool(KEEP_METALS),
    "METAL_ELEMENTS_LIST": METAL_ELEMENTS_LIST,
    "METAL_TO_ANALYZE": METAL_TO_ANALYZE,
    "SOLVATE_SYSTEM": bool(SOLVATE_SYSTEM),
    "WATER_PADDING_ANG": float(WATER_PADDING_ANG),
    "NEUTRALIZE_SYSTEM": bool(NEUTRALIZE_SYSTEM),
    "EXTRA_NA": int(EXTRA_NA),
    "EXTRA_CL": int(EXTRA_CL),
    "TEMPERATURE_K": float(TEMPERATURE_K),
    "PRESSURE_BAR": float(PRESSURE_BAR),
    "DT_FS": float(DT_FS),
    "FRICTION_PS": float(FRICTION_PS),
    "EQUIL_STEPS_TOTAL": int(EQUIL_STEPS_TOTAL),
    "PROD_STEPS_TO_RUN": int(PROD_STEPS_TO_RUN),
}
CONFIG_JSON.write_text(json.dumps(config, indent=2, ensure_ascii=False))

print("WORKDIR:", WORKDIR)
print("Protein input:", PROT_IN)
print("Ligand mode:", LIGAND_KIND)
print("Ligand input:", LIG_IN)
print("System basename:", SYSTEM_BASENAME)
print("Saved config:", CONFIG_JSON)


In [ ]:
#@title 3) Copy inputs into WORKDIR (if uploaded to /content) ✅
from pathlib import Path
import shutil

WORKDIR.mkdir(parents=True, exist_ok=True)

src_candidates = [
    (Path(f"/content/{PROTEIN_INPUT_NAME}"), PROT_IN),
    (Path(f"/content/{LIGAND_INPUT_NAME}"),  LIG_IN),
]

for src, dst in src_candidates:
    if src.exists() and (not dst.exists() or dst.stat().st_size != src.stat().st_size):
        shutil.copy2(src, dst)
        print("Copied:", src, "->", dst)

assert PROT_IN.exists(), f"Missing protein file in WORKDIR: {PROT_IN.name}"
assert LIG_IN.exists(),  f"Missing ligand file in WORKDIR: {LIG_IN.name}"

print("Protein present:", PROT_IN, "| size:", PROT_IN.stat().st_size)
print("Ligand present :", LIG_IN,  "| size:", LIG_IN.stat().st_size)


In [ ]:
!apt-get update -qq
!apt-get install -y openbabel
!which obabel
!obabel -V

In [ ]:
#@title 4) Ligand Parameterization (Force Residue Name: LIG) ✅
import subprocess, textwrap, os
from pathlib import Path

# Принудительно используем имя LIG для максимальной совместимости
res_name = "LIG"
print(f"--- Parameterizing ligand. Force residue name: {res_name} ---")

LIGAND_MOL2 = WORKDIR / f"{SYSTEM_BASENAME}_ligand_gaff2.mol2"
LIGAND_FRCMOD = WORKDIR / f"{SYSTEM_BASENAME}_ligand_gaff2.frcmod"
LIGAND_LIB = WORKDIR / f"{SYSTEM_BASENAME}_ligand_gaff2.lib"

# --- ШАГ 0: ПЕРЕИМЕНОВАНИЕ ОСТАТКА В MOL2 ---
# Это гарантирует, что в .lib файле имя будет 'LIG'
with open(LIGAND_MOL2, 'r') as f:
    mol2_content = f.read()
# Заменяем имя остатка (обычно это 8-я колонка в секции ATOM или имя в начале)
# Мы просто заменим UNK на LIG по всему файлу, это безопасно для mol2
mol2_content = mol2_content.replace("UNK", "LIG")
with open(LIGAND_MOL2, 'w') as f:
    f.write(mol2_content)

# 1. Генерация frcmod
subprocess.run(
    f"parmchk2 -i {LIGAND_MOL2.as_posix()} -f mol2 -o {LIGAND_FRCMOD.as_posix()} -s {GAFF_VERSION}",
    shell=True, check=True
)

# 2. Создание библиотеки (.lib)
lib_gen_in = WORKDIR / "gen_lib.in"
lib_gen_in.write_text(textwrap.dedent(f"""
    source leaprc.{GAFF_VERSION}
    {res_name} = loadMol2 {LIGAND_MOL2.as_posix()}
    saveOff {res_name} {LIGAND_LIB.as_posix()}
    quit
"""))

subprocess.run(f"tleap -f {lib_gen_in.as_posix()} > {WORKDIR}/gen_lib.log 2>&1", shell=True, check=True)

if LIGAND_LIB.exists():
    print(f"✅ Успех: Библиотека создана с именем остатка '{res_name}'")
else:
    raise RuntimeError("ОШИБКА: .lib файл не создался.")

In [ ]:
#@title 5) Protein prep (pdb4amber on protein only) ✅
from pathlib import Path
import shutil, subprocess

WD = WORKDIR
prot_in = PROT_IN
prot_out = PROT_AMBER_PDB

r = subprocess.run("which pdb4amber", shell=True, capture_output=True, text=True)
if r.returncode == 0:
    cmd = f"pdb4amber -i {prot_in.name} -o {prot_out.name} --nohyd --reduce"
    subprocess.run(cmd, shell=True, cwd=str(WD), check=True)
    print("✅ pdb4amber wrote:", prot_out)
else:
    shutil.copy2(prot_in, prot_out)
    print("⚠️ pdb4amber not found; copied input protein to:", prot_out)


In [ ]:
from pathlib import Path
import subprocess

# пути
ligand_mol2 = Path(LIGAND_MOL2)
ligand_pdb = Path(WD) / f"{SYSTEM_BASENAME}_ligand_LEaP_view.pdb"

def run(cmd):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout)
    if r.stderr:
        print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

# если PDB ещё не создан — делаем его из mol2
if not ligand_pdb.exists():
    print("Ligand PDB is missing, creating it from MOL2...")
    run(f'obabel "{ligand_mol2}" -O "{ligand_pdb}"')

print("Ligand PDB:", ligand_pdb)
print("Exists:", ligand_pdb.exists())

In [ ]:
#@title 6) Merge protein + ligand into prepared complex PDB ✅
from pathlib import Path

WD = WORKDIR
protein_pdb = PROT_AMBER_PDB
ligand_pdb  = LIGAND_LEAP_PDB

assert protein_pdb.exists(), f"Missing protein PDB: {protein_pdb}"
assert ligand_pdb.exists(),  f"Missing ligand PDB: {ligand_pdb}"

raw_out = RAW_COMPLEX_PDB
prepared = PREPARED_COMPLEX_PDB

PHOS_ATOMS = {"P","OP1","OP2","O1P","O2P","OP3"}

def is_h(ln):
    an = ln[12:16].strip().upper()
    elem = ln[76:78].strip().upper() if len(ln) >= 78 else ""
    return elem == "H" or an.startswith("H")

def is_metal_line(ln):
    if not ln.startswith(("ATOM","HETATM")):
        return False
    resn = ln[17:20].strip().upper()
    an   = ln[12:16].strip().upper()
    elem = (ln[76:78].strip().upper() if len(ln) >= 78 else "")
    tokens = {resn, an, elem}
    return any(tok in METAL_ELEMENTS_LIST for tok in tokens if tok)

def read_atoms(p: Path):
    out = []
    for ln in p.read_text(errors="ignore").splitlines():
        if ln.startswith(("ATOM","HETATM")):
            out.append(ln[:80])
    return out

prot_lines = read_atoms(protein_pdb)
lig_lines  = read_atoms(ligand_pdb)

serial = 1
merged = []
for block in (prot_lines, ["TER"], lig_lines, ["TER"], ["END"]):
    for ln in block:
        if ln in {"TER","END"}:
            merged.append(ln)
        else:
            merged.append(f"{ln[:6]}{serial:5d}{ln[11:]}")
            serial += 1

raw_out.write_text("\n".join(merged) + "\n")
print("Wrote merged complex:", raw_out)

cleaned = []
for ln in raw_out.read_text(errors="ignore").splitlines():
    rec = ln[:6].strip()

    if rec == "TER":
        cleaned.append("TER")
        continue
    if rec == "END":
        continue
    if rec not in {"ATOM", "HETATM"}:
        continue

    if is_h(ln):
        continue

    rn = ln[17:20].strip().upper()
    at = ln[12:16].strip().upper()

    if LIGAND_KIND == "nucleic_acid":
        if rn.endswith("5") and at in PHOS_ATOMS:
            continue
        if at in MARKER_ATOMS:
            continue

    cleaned.append(ln[:80])

collapsed = []
for ln in cleaned:
    if ln == "TER" and collapsed and collapsed[-1] == "TER":
        continue
    collapsed.append(ln)

if not collapsed or collapsed[-1] != "TER":
    collapsed.append("TER")
collapsed.append("END")

prepared.write_text("\n".join(collapsed) + "\n")
print("✅ Prepared complex for LEaP:", prepared)


In [ ]:
#@title FIX: preserve metals from original inputs if requested ✅
from pathlib import Path
import json

WD = WORKDIR
prep = PREPARED_COMPLEX_PDB

assert prep.exists(), f"Missing prepared complex: {prep}"

def is_target_metal(ln: str) -> bool:
    if not ln.startswith(("ATOM","HETATM")):
        return False
    resn = ln[17:20].strip().upper()
    an   = ln[12:16].strip().upper()
    elem = (ln[76:78].strip().upper() if len(ln) >= 78 else "")
    tokens = {resn, an, elem}
    return any(tok in METAL_ELEMENTS_LIST for tok in tokens if tok)

if not KEEP_METALS:
    print("KEEP_METALS=False -> skipping metal reinsertion.")
else:
    metal_lines = []
    for src in [PROT_IN, LIG_IN]:
        if not src.exists():
            continue
        for ln in src.read_text(errors="ignore").splitlines():
            if is_target_metal(ln):
                metal_lines.append(ln[:80])

    unique_metals = []
    seen = set()
    for ln in metal_lines:
        key = ln[12:16].strip(), ln[17:20].strip(), ln[21].strip(), ln[22:26].strip(), ln[30:54].strip(), ln[76:78].strip()
        if key not in seen:
            seen.add(key)
            unique_metals.append(ln)

    prep_lines = prep.read_text(errors="ignore").splitlines()
    already = [ln for ln in prep_lines if is_target_metal(ln)]

    print("Metals found in source inputs:", len(unique_metals))
    print("Metals already present in prepared complex:", len(already))

    if len(unique_metals) == 0:
        print("No target metals detected in the original inputs.")
    else:
        out = []
        inserted = False
        for ln in prep_lines:
            if ln.strip() == "END" and not inserted:
                for m in unique_metals:
                    out.append(m)
                    out.append("TER")
                inserted = True
            out.append(ln)
        if not inserted:
            for m in unique_metals:
                out.append(m)
                out.append("TER")
            out.append("END")
        prep.write_text("\n".join(out) + "\n")
        print("✅ Metals reinserted into prepared complex if they were missing.")


In [ ]:
#@title 7) Create fixed leaprc for TIP3P + ions (robust) ✅
from pathlib import Path

WD = WORKDIR
parm_dir = Path("/usr/local/dat/leap/parm")

mono_candidates = ["frcmod.ionsjc_tip3p", "frcmod.ions1lm_126_tip3p", "frcmod.ions1lsm_126_tip3p"]
multi_candidates = ["frcmod.ions234lm_126_tip3p", "frcmod.ions234lsm_126_tip3p", "frcmod.ions234lm_126_spce"]

mono = next((x for x in mono_candidates if (parm_dir/x).exists()), None)
multi = next((x for x in multi_candidates if (parm_dir/x).exists()), None)

print("Monovalent ion frcmod:", mono)
print("Multivalent ion frcmod:", multi)

if mono is None and multi is None:
    raise RuntimeError("No ion frcmod files found in /usr/local/dat/leap/parm")

fixed = WD / "leaprc.water.tip3p.fixed"
lines = ["source leaprc.water.tip3p"]
if mono:
    lines.append(f"loadamberparams {parm_dir/mono}")
if multi:
    lines.append(f"loadamberparams {parm_dir/multi}")
fixed.write_text("\n".join(lines) + "\n")
print("Wrote:", fixed)


In [ ]:
#@title 8) Build AMBER topology with Atom Sync ✅
import subprocess, textwrap
from pathlib import Path

# --- 1. ПРОВЕРКА ИМЕН АТОМОВ (Синхронизация PDB и MOL2) ---
def sync_ligand_atoms(pdb_path, mol2_path, out_pdb):
    # Извлекаем имена атомов из MOL2 (секция @<TRIPOS>ATOM)
    mol2_atoms = []
    with open(mol2_path, 'r') as f:
        in_atom_section = False
        for line in f:
            if "@<TRIPOS>ATOM" in line:
                in_atom_section = True; continue
            if "@<TRIPOS>BOND" in line or "@<TRIPOS>SUBSTRUCTURE" in line:
                in_atom_section = False; break
            if in_atom_section:
                parts = line.split()
                if len(parts) > 1: mol2_atoms.append(parts[1])

    # Читаем PDB и заменяем UNK -> LIG + подставляем имена атомов
    new_lines = []
    lig_atom_idx = 0
    with open(pdb_path, 'r') as f:
        for line in f:
            if ("HETATM" in line or "ATOM" in line) and ("UNK" in line or "LIG" in line):
                # Форматируем строку PDB: ставим имя из MOL2 и остаток LIG
                name = mol2_atoms[lig_atom_idx] if lig_atom_idx < len(mol2_atoms) else "H"
                # Колонки PDB: AtomName(12-16), ResName(17-20)
                l = list(line)
                l[12:16] = f"{name:<4}"[:4]
                l[17:20] = "LIG"
                new_lines.append("".join(l))
                lig_atom_idx += 1
            else:
                new_lines.append(line)

    with open(out_pdb, 'w') as f:
        f.writelines(new_lines)
    print(f"✅ Синхронизировано {lig_atom_idx} атомов лиганда")

# Пути
LIGAND_MOL2 = WORKDIR / f"{SYSTEM_BASENAME}_ligand_gaff2.mol2"
LIGAND_FRCMOD = WORKDIR / f"{SYSTEM_BASENAME}_ligand_gaff2.frcmod"
LIGAND_LIB = WORKDIR / f"{SYSTEM_BASENAME}_ligand_gaff2.lib"
SYNC_PDB = WORKDIR / "final_sync_complex.pdb"

# Запускаем синхронизацию
sync_ligand_atoms(PREPARED_COMPLEX_PDB, LIGAND_MOL2, SYNC_PDB)

# --- 2. ЗАПУСК TLEAP ---
SYSTEM_PRMTOP = WORKDIR / f"SYS_{SYSTEM_BASENAME}.prmtop"
SYSTEM_INPCRD = WORKDIR / f"SYS_{SYSTEM_BASENAME}.crd"
LEAP_IN = WORKDIR / "build.leap.in"

leap_script = textwrap.dedent(f"""
    set default PBRadii mbondi3
    source leaprc.protein.ff14SB
    source leaprc.water.tip3p
    source leaprc.{GAFF_VERSION}
    loadAmberParams {LIGAND_FRCMOD.as_posix()}
    loadOff {LIGAND_LIB.as_posix()}
    complex = loadPdb {SYNC_PDB.as_posix()}
    {"solvateBox complex TIP3PBOX " + str(WATER_PADDING_ANG) if SOLVATE_SYSTEM else ""}
    addIonsRand complex Na+ 0
    addIonsRand complex Cl- 0
    saveAmberParm complex {SYSTEM_PRMTOP.as_posix()} {SYSTEM_INPCRD.as_posix()}
    quit
""")

LEAP_IN.write_text(leap_script)
subprocess.run(f"tleap -f {LEAP_IN.as_posix()} > {WORKDIR}/tleap_final.log 2>&1", shell=True)

if SYSTEM_PRMTOP.exists():
    print(f"✅ Топология готова: {SYSTEM_PRMTOP.name}")
else:
    print("❌ Ошибка в tleap. Проверь 'tleap_final.log'")

In [ ]:
#@title Cleanup old checkpoints (required after topology change) ✅
from pathlib import Path
WD = WORKDIR
for fn in [f"{EQUIL_JOB}.chk", f"{PROD_JOB}.chk"]:
    p = WD / fn
    if p.exists():
        p.unlink()
        print("Removed old checkpoint:", p.name)


In [ ]:
#@title 9) Running Energy Minimization & Equilibration (Universal Platform) ✅
from openmm import app, unit
import openmm as mm
import sys

# Параметры конвертации (используем твои переменные)
dcd_steps = int((EQUIL_DCD_PS * 1000) / DT_FS)
log_steps = int((EQUIL_LOG_PS * 1000) / DT_FS)
chk_steps = int((EQUIL_CHK_PS * 1000) / DT_FS)

print(f"--- Starting Equilibration for: {SYSTEM_BASENAME} ---")

# Загрузка топологии и координат
prmtop = app.AmberPrmtopFile(str(SYSTEM_PRMTOP))
inpcrd = app.AmberInpcrdFile(str(SYSTEM_CRD))

# Создание системы
system = prmtop.createSystem(
    app.PME if SOLVATE_SYSTEM else app.NoCutoff,
    nonbondedCutoff=1.0*unit.nanometers,
    constraints=app.HBonds
)
system.addForce(mm.MonteCarloBarostat(PRESSURE_BAR*unit.bar, TEMPERATURE_K*unit.kelvin, 25))

# Интегратор
integrator = mm.LangevinMiddleIntegrator(TEMPERATURE_K*unit.kelvin, FRICTION_PS/unit.picosecond, DT_FS*unit.femtoseconds)

# ---------------------------------------------------------
# СТРОГАЯ СИНХРОНИЗАЦИЯ ПЛАТФОРМЫ (КАК В ЯЧЕЙКЕ 10)
# ---------------------------------------------------------
properties = {}
try:
    platform = mm.Platform.getPlatformByName("CUDA")
    properties["Precision"] = "mixed"
    print("Platform: CUDA (Mixed Precision) - SYNCED")
except Exception:
    platform = mm.Platform.getPlatformByName("CPU")
    print("Platform: CPU - SYNCED")

simulation = app.Simulation(prmtop.topology, system, integrator, platform, properties)
simulation.context.setPositions(inpcrd.positions)
if inpcrd.boxVectors is not None:
    simulation.context.setPeriodicBoxVectors(*inpcrd.boxVectors)

print("Performing energy minimization...")
simulation.minimizeEnergy(maxIterations=1000)
print("✅ Minimization finished.")

simulation.context.setVelocitiesToTemperature(TEMPERATURE_K*unit.kelvin)

# Репортеры
simulation.reporters.append(app.StateDataReporter(
    sys.stdout, log_steps, step=True, potentialEnergy=True,
    temperature=True, density=True, speed=True, progress=True,
    totalSteps=EQUIL_STEPS_TOTAL
))
dcd_path = WORKDIR / f"{EQUIL_JOB}.dcd"
simulation.reporters.append(app.DCDReporter(str(dcd_path), dcd_steps))

print(f"Running NPT equilibration for {EQUIL_STEPS_TOTAL} steps...")
simulation.step(EQUIL_STEPS_TOTAL)

# ---------------------------------------------------------
# ДВОЙНОЕ СОХРАНЕНИЕ: CHK (быстрый) и XML (универсальный)
# ---------------------------------------------------------
chk_path = WORKDIR / f"{EQUIL_JOB}.chk"
xml_path = WORKDIR / f"{EQUIL_JOB}.xml"

simulation.saveCheckpoint(str(chk_path))
simulation.saveState(str(xml_path)) # <- Это спасет нас от конфликта платформ

print(f"✅ Equilibration complete! State saved to .chk and .xml")

In [ ]:
#@title 🧹 Surgical Cleanup: Reset Production State
import os, glob
from pathlib import Path

print(f"--- Начинаем очистку битых файлов Production ---")
print(f"Папка: {WORKDIR}")

# Ищем все файлы, которые начинаются с имени твоего PROD_JOB (например, prot_lig_prod)
prod_files_drive = glob.glob(str(WORKDIR / f"{PROD_JOB}*"))
prod_files_tmp = glob.glob(f"/content/{PROD_JOB}*")

all_targets = prod_files_drive + prod_files_tmp

if not all_targets:
    print("✅ Битых файлов не найдено. Система чиста.")
else:
    for f_path in all_targets:
        try:
            os.remove(f_path)
            print(f"🗑 Удален битый файл: {Path(f_path).name}")
        except Exception as e:
            print(f"⚠️ Не удалось удалить {f_path}: {e}")

print("✅ Очистка завершена. Теперь можно безопасно запускать 10-ю ячейку.")

In [ ]:
#@title 10) Production MD (Chunked Trajectories - PRO FIX) ✅
import os, shutil, glob
from pathlib import Path
from openmm.app import AmberPrmtopFile, AmberInpcrdFile, Simulation, PME, HBonds, PDBFile, DCDReporter, StateDataReporter, CheckpointReporter
import openmm as mm
from openmm import unit
import sys

# Настройки путей
WD = WORKDIR
TMP = Path("/content")

job = PROD_JOB
# Базовые пути на Google Drive (траекторию определим динамически)
log_drive  = WD / f"{job}.log"
chk_drive  = WD / f"{job}.chk"
xml_drive  = WD / f"{job}.xml"
last_drive = WD / f"{job}_last.pdb"

# Локальные пути
log_tmp  = TMP / f"{job}.log"
chk_tmp  = TMP / f"{job}.chk"
xml_tmp  = TMP / f"{job}.xml"
last_tmp = TMP / f"{job}_last.pdb"

equil_chk = WD / f"{EQUIL_JOB}.chk"
equil_xml = WD / f"{EQUIL_JOB}.xml"

# Параметры времени
dt = DT_FS * unit.femtoseconds
steps_dcd = int(round((PROD_DCD_PS * unit.picoseconds) / dt))
steps_log = int(round((PROD_LOG_PS * unit.picoseconds) / dt))
steps_chk = int(round((PROD_CHK_PS * unit.picoseconds) / dt))

print("> Production details:")
print(f"  job: {job}")
print(f"  write every: {PROD_DCD_PS} ps ({steps_dcd} steps)")

prmtop = AmberPrmtopFile(str(SYSTEM_PRMTOP))
inpcrd = AmberInpcrdFile(str(SYSTEM_CRD))
system = prmtop.createSystem(PME, nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds, rigidWater=True, removeCMMotion=True)
system.addForce(mm.MonteCarloBarostat(PRESSURE_BAR * unit.bar, TEMPERATURE_K * unit.kelvin, 25))
integrator = mm.LangevinMiddleIntegrator(TEMPERATURE_K * unit.kelvin, FRICTION_PS / unit.picosecond, dt)

properties = {"Precision": "mixed"}
try:
    platform = mm.Platform.getPlatformByName("CUDA")
    print("Platform: CUDA (Mixed Precision) - SYNCED")
except Exception:
    platform = mm.Platform.getPlatformByName("CPU")
    print("Platform: CPU - SYNCED")

simulation = Simulation(prmtop.topology, system, integrator, platform, properties)

def try_load_state(chk_file, xml_file):
    if chk_file.exists():
        try:
            with open(chk_file, "rb") as f:
                simulation.context.loadCheckpoint(f.read())
            return True
        except: pass
    if xml_file.exists():
        try:
            simulation.loadState(str(xml_file))
            return True
        except: pass
    return False

resumed = False
if chk_drive.exists():
    print("> Found existing Production checkpoint. Resuming...")
    resumed = try_load_state(chk_drive, xml_drive)

# --- УМНОЕ РАЗБИЕНИЕ ТРАЕКТОРИЙ (TRAJECTORY CHUNKING) ---
dcd_name = f"{job}.dcd"

if resumed:
    # Ищем свободный суффикс для файла траектории, чтобы не трогать битый заголовок
    part = 2
    while (WD / f"{job}_part{part}.dcd").exists():
        part += 1
    dcd_name = f"{job}_part{part}.dcd"
    print(f"⚠️ Binary append disabled to prevent header corruption.")
    print(f"🟢 Writing coordinates to new chunk: {dcd_name}")

    # Для текстового лога append безопасен, но нужно скопировать старый с Диска
    if log_drive.exists():
        shutil.copy2(log_drive, log_tmp)
else:
    print("> No Production state found. Loading from Equilibration...")
    resumed = try_load_state(equil_chk, equil_xml)
    if not resumed:
        print("> 🚨 Starting from scratch (PDB coordinates).")
        simulation.context.setPositions(inpcrd.positions)
        simulation.minimizeEnergy(maxIterations=1000)
        simulation.context.setVelocitiesToTemperature(TEMPERATURE_K * unit.kelvin)

# Определяем пути для текущего чанка траектории
dcd_drive = WD / dcd_name
dcd_tmp = TMP / dcd_name

target_total = simulation.currentStep + PROD_STEPS_TO_RUN
log_mode = "a" if resumed else "w"

simulation.reporters.clear()

# ВАЖНО: всегда создаем новый файл DCD для текущего чанка (append=False)
simulation.reporters.append(DCDReporter(str(dcd_tmp), steps_dcd, append=False))

simulation.reporters.append(StateDataReporter(
    sys.stdout, steps_log, step=True, time=True, potentialEnergy=True,
    temperature=True, progress=True, remainingTime=True, speed=True,
    totalSteps=target_total
))
log_handle = open(log_tmp, log_mode)
simulation.reporters.append(StateDataReporter(
    log_handle, steps_log, step=True, time=True, potentialEnergy=True,
    temperature=True, density=True, separator="\t"
))

simulation.reporters.append(CheckpointReporter(str(chk_tmp), steps_chk))

print(f"> Current step: {simulation.currentStep}, Target: {target_total}")

try:
    simulation.step(PROD_STEPS_TO_RUN)
except Exception as e:
    print(f"❌ Simulation Error: {e}")
finally:
    log_handle.close()
    state = simulation.context.getState(getPositions=True)
    with open(last_tmp, "w") as f:
        PDBFile.writeFile(simulation.topology, state.getPositions(), f)
    simulation.saveState(str(xml_tmp))

    print("> Syncing files with Google Drive...")
    # Синхронизируем все файлы, включая наш новый DCD чанк
    for src, dst in [(dcd_tmp, dcd_drive), (log_tmp, log_drive), (chk_tmp, chk_drive), (xml_tmp, xml_drive), (last_tmp, last_drive)]:
        if src.exists():
            shutil.copy2(src, dst)

print("✅ Production finished and synced.")

In [ ]:
#@title 11) Trajectory Analysis (Turbo SSD Cache Version) 🚀
import mdtraj as md
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os, glob, re, shutil
from pathlib import Path

EXCLUDE_TAIL = True

print("--- Запуск оптимизированного анализа (Локальный SSD) ---")
try: os.sync()
except: pass

# 1. Поиск файлов на Google Диске
paths_to_search = [WORKDIR, Path("/content")]
dcd_files = []
target_pattern = f"{PROD_JOB}*.dcd"

for folder in paths_to_search:
    if folder.exists():
        dcd_files.extend(glob.glob(str(folder / target_pattern)))
dcd_files = list(set(dcd_files))

if not dcd_files:
    raise FileNotFoundError(f"Файлы {target_pattern} не найдены.")

def chunk_sort_key(filepath):
    match = re.search(r'part(\d+)', Path(filepath).name)
    return int(match.group(1)) if match else 1
dcd_files_sorted = sorted(dcd_files, key=chunk_sort_key)

# 2. Валидация
valid_dcd_files = []
for f in dcd_files_sorted:
    if os.path.getsize(f) / (1024 * 1024) > 0.1:
        try:
            with md.formats.DCDTrajectoryFile(f) as t: len(t)
            valid_dcd_files.append(f)
        except: pass

if not valid_dcd_files:
    raise ValueError("Нет целых файлов DCD для анализа.")

# 3. УСКОРЕНИЕ: Копирование на локальный SSD Colab
local_cache_dir = Path("/content/md_local_cache")
local_cache_dir.mkdir(exist_ok=True)

# Копируем топологию (.prmtop)
top_path = str(SYSTEM_PRMTOP)
local_top_path = local_cache_dir / Path(top_path).name
print(f"⚡ Копирую топологию на локальный SSD...")
shutil.copy(top_path, local_top_path)

# Копируем траектории (.dcd)
local_dcd_files = []
for f in valid_dcd_files:
    loc_f = local_cache_dir / Path(f).name
    print(f"⚡ Локальное кеширование: {Path(f).name} -> SSD...")
    shutil.copy(f, loc_f)
    local_dcd_files.append(str(loc_f))

# 4. МГНОВЕННАЯ ЗАГРУЗКА ИЗ ЛОКАЛЬНОГО КЕША
print(f"\n🚀 Считывание данных из сверхбыстрого кеша...")
traj = md.load(local_dcd_files, top=str(local_top_path))
print(f"✅ Успешно загружено: {traj.n_frames} кадров.")

# 5. ФИЗИЧЕСКИЕ РАСЧЕТЫ
if EXCLUDE_TAIL:
    protein_ca = traj.topology.select("protein and name CA and residue 0 to 295")
    print("✂️ Анализ выполняется для стабильного кора белка (остатки 0-295).")
else:
    protein_ca = traj.topology.select("protein and name CA")
    print("⚠️ Анализ выполняется для полного белка.")

ligand_heavy = traj.topology.select(f"resname {SMALL_MOL_RESNAME} and not element H")
ligand_all = traj.topology.select(f"resname {SMALL_MOL_RESNAME}")

if len(ligand_heavy) == 0:
    ligand_heavy = ligand_all

traj.superpose(traj, 0, atom_indices=protein_ca)
time_ns = np.arange(traj.n_frames) * (PROD_DCD_PS / 1000.0)

rmsd_prot = md.rmsd(traj, traj, 0, atom_indices=protein_ca) * 10
rmsd_lig = md.rmsd(traj, traj, 0, atom_indices=ligand_heavy) * 10 if len(ligand_heavy) > 0 else np.zeros(traj.n_frames)
rmsf_ca = md.rmsf(traj, traj, 0, atom_indices=protein_ca) * 10
ca_indices = np.arange(len(protein_ca)) + 1

print("Расчет расстояния между центрами масс (COM distance)...")
com_protein = md.compute_center_of_mass(traj.atom_slice(protein_ca))
com_ligand = md.compute_center_of_mass(traj.atom_slice(ligand_all))
com_distance = np.sqrt(np.sum((com_protein - com_ligand)**2, axis=1))

# 6. ОТРИСОВКА В ЕДИНОМ СТИЛЕ (Глубокий синий)
plt.rcParams.update({'font.size': 11, 'font.family': 'sans-serif'})
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
MAIN_BLUE = '#1f77b4'
grid_style = dict(linestyle='-', linewidth=0.5, color='#e6e6e6', alpha=0.7)

# Protein Cα RMSD
axs[0, 0].plot(time_ns, rmsd_prot, color=MAIN_BLUE, linewidth=1.2)
axs[0, 0].set_title("Protein Cα RMSD")
axs[0, 0].set_xlabel("Time (ns)")
axs[0, 0].set_ylabel("RMSD (Å)")
axs[0, 0].grid(**grid_style)
axs[0, 0].set_xlim(left=0)

# Ligand heavy-atom RMSD
axs[0, 1].plot(time_ns, rmsd_lig, color=MAIN_BLUE, linewidth=1.2)
axs[0, 1].set_title("Ligand heavy-atom RMSD")
axs[0, 1].set_xlabel("Time (ns)")
axs[0, 1].set_ylabel("RMSD (Å)")
axs[0, 1].grid(**grid_style)
axs[0, 1].set_xlim(left=0)

# Protein Cα RMSF
axs[1, 0].plot(ca_indices, rmsf_ca, color=MAIN_BLUE, linewidth=1.2)
axs[1, 0].set_title("Protein Cα RMSF")
axs[1, 0].set_xlabel("Cα index")
axs[1, 0].set_ylabel("RMSF (Å)")
axs[1, 0].grid(**grid_style)
axs[1, 0].set_xlim(left=0)

# Ligand-protein COM distance
axs[1, 1].plot(time_ns, com_distance, color=MAIN_BLUE, linewidth=1.2)
axs[1, 1].set_title("Ligand-protein COM distance")
axs[1, 1].set_xlabel("Time (ns)")
axs[1, 1].set_ylabel("Distance (nm)")
axs[1, 1].grid(**grid_style)
axs[1, 1].set_xlim(left=0)

plt.tight_layout()
plot_path = WORKDIR / f"{SYSTEM_BASENAME}_unified_summary.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')

# Очистка локального кеша, чтобы не забивать место в Colab
shutil.rmtree(local_cache_dir, ignore_errors=True)

print(f"📊 График успешно сохранен на ваш Диск в: {plot_path}")
plt.show()

In [ ]:
#@title 12) 12) Protein Cα RMSD + distribution ✅
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import mdtraj as md

prot_ca = traj.topology.select("protein and name CA")
rmsd_prot = md.rmsd(traj, traj, 0, atom_indices=prot_ca)  # nm
xA = rmsd_prot * 10.0
print(f"Cα RMSD: mean={xA.mean():.2f} Å | max={xA.max():.2f} Å")

fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(xA)
ax.set_xlabel("Frame"); ax.set_ylabel("Cα RMSD (Å)"); ax.set_title("Protein Cα RMSD")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6,4))
ax.hist(xA, bins=30, density=True, alpha=0.5, edgecolor='k')
grid = np.linspace(xA.min(), xA.max(), 400)
ax.plot(grid, gaussian_kde(xA)(grid), linewidth=2, label="KDE")
ax.set_xlabel("Cα RMSD (Å)"); ax.set_ylabel("Probability density"); ax.set_title("Cα RMSD distribution")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
#@title 13) Ligand RMSD (NA branch or small-molecule branch) ✅
import numpy as np
import mdtraj as md

traj_al = traj.superpose(traj, 0, atom_indices=prot_ca)

if LIGAND_KIND == "nucleic_acid":
    NA_NAMES = {
        "A","C","G","U","DA","DC","DG","DT","RA","RC","RG","RU",
        "DA5","DG5","DT3","DC3","A5","C3","G5",
        MODIFIED_NA_RESNAME.upper()
    }
    lig_heavy = np.array([
        a.index for a in traj_al.topology.atoms
        if a.residue.name.strip().upper() in NA_NAMES and a.element.symbol != "H"
    ], dtype=int)
    assert len(lig_heavy) > 0, "NA ligand atoms not found (check resnames in topology)."
    rmsd_lig = md.rmsd(traj_al, traj_al[0], atom_indices=lig_heavy)
    print("NA heavy atoms:", len(lig_heavy))
    print(f"NA RMSD: mean={float(rmsd_lig.mean()):.3f} nm | max={float(rmsd_lig.max()):.3f} nm")

elif LIGAND_KIND == "small_molecule":
    target = SMALL_MOL_RESNAME.strip().upper()
    lig_heavy = np.array([
        a.index for a in traj_al.topology.atoms
        if a.residue.name.strip().upper() == target and a.element.symbol != "H"
    ], dtype=int)
    assert len(lig_heavy) > 0, f"Small-molecule ligand atoms not found for residue name {target}."
    rmsd_lig = md.rmsd(traj_al, traj_al[0], atom_indices=lig_heavy)
    print("Ligand heavy atoms:", len(lig_heavy))
    print(f"Small-molecule RMSD: mean={float(rmsd_lig.mean()):.3f} nm | max={float(rmsd_lig.max()):.3f} nm")

else:
    raise ValueError(f"Unknown LIGAND_KIND: {LIGAND_KIND}")


In [ ]:
#@title 14) 14) Protein Cα RMSF ✅
import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md

rmsf = md.rmsf(traj_al, traj_al[0], atom_indices=prot_ca)  # nm
fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(rmsf*10.0)
ax.set_xlabel("Cα index"); ax.set_ylabel("RMSF (Å)"); ax.set_title("14) Protein Cα RMSF")
plt.tight_layout(); plt.show()

In [ ]:
#@title 15) Metal coordination sanity (skips gracefully if no metal) ✅
import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md

top = traj_al.topology

def atom_matches_target(a, target):
    if target == "auto":
        return (
            (a.element is not None and a.element.symbol.upper() in METAL_ELEMENTS_LIST)
            or a.name.strip().upper() in METAL_ELEMENTS_LIST
            or a.residue.name.strip().upper() in METAL_ELEMENTS_LIST
        )
    tgt = target.strip().upper()
    return (
        (a.element is not None and a.element.symbol.upper() == tgt)
        or a.name.strip().upper() == tgt
        or a.residue.name.strip().upper() == tgt
    )

metal_idx = np.array([a.index for a in top.atoms if atom_matches_target(a, METAL_TO_ANALYZE)], dtype=int)

if len(metal_idx) == 0:
    print("No requested metal found in trajectory. Skipping coordination sanity check.")
else:
    metal = int(metal_idx[0])
    metal_atom = list(top.atoms)[metal]
    print("Using metal atom:", metal, metal_atom)

    carbox_o = top.select("protein and (name OD1 or name OD2 or name OE1 or name OE2)")
    if len(carbox_o) == 0:
        print("No protein carboxylate oxygens found. Nothing to analyze.")
    else:
        pairs = np.array([[metal, i] for i in carbox_o], dtype=int)
        d = md.compute_distances(traj_al, pairs)
        dmin = d.min(axis=1)
        print(f"Min metal–O(protein): mean={dmin.mean():.3f} nm | max={dmin.max():.3f} nm")

        fig, ax = plt.subplots(figsize=(6,4))
        ax.hist(dmin, bins=30, density=True, alpha=0.5, edgecolor="k")
        ax.set_xlabel("Min metal–O(protein) distance (nm)")
        ax.set_ylabel("Probability density")
        ax.set_title("Metal–protein O distance distribution")
        plt.tight_layout()
        plt.show()


In [ ]:
#@title DEBUG: where is the metal? (inputs -> prepared -> tleap outputs -> prmtop -> traj) ✅
from pathlib import Path
import parmed as pmd
import mdtraj as md

WD = WORKDIR

files = {
    "protein_input.pdb": PROT_IN,
    "protein_amber.pdb": PROT_AMBER_PDB,
    "prepared_complex.pdb": PREPARED_COMPLEX_PDB,
    "tleap_SYS.pdb": SYSTEM_PDB,
    "prmtop": SYSTEM_PRMTOP,
    "prod_dcd": WD / f"{PROD_JOB}.dcd",
}

def grep_target_metal(p: Path):
    if not p.exists():
        return False, None
    for ln in p.read_text(errors="ignore").splitlines():
        if not ln.startswith(("ATOM","HETATM")):
            continue
        resn = ln[17:20].strip().upper()
        an   = ln[12:16].strip().upper()
        elem = (ln[76:78].strip().upper() if len(ln) >= 78 else "")
        tokens = {resn, an, elem}
        if any(tok in METAL_ELEMENTS_LIST for tok in tokens if tok):
            return True, ln[:80]
    return False, None

print("---- PDB checks ----")
for k, p in files.items():
    if p.suffix.lower() == ".pdb":
        ok, line = grep_target_metal(p)
        print(f"{k:22s} exists={p.exists()}  metal={ok}")
        if line:
            print("  example:", line)

print("\n---- prmtop check ----")
prm = files["prmtop"]
if prm.exists() and prm.stat().st_size > 2000:
    parm = pmd.load_file(str(prm))
    metal_atoms = [
        a for a in parm.atoms
        if a.name.strip().upper() in METAL_ELEMENTS_LIST
        or a.residue.name.strip().upper() in METAL_ELEMENTS_LIST
    ]
    print("Target metal-like atoms in prmtop:", len(metal_atoms))
else:
    print("prmtop missing/too small")

print("\n---- traj check ----")
dcd = files["prod_dcd"]
if dcd.exists() and dcd.stat().st_size > 1_000_000 and prm.exists():
    t = md.load_dcd(str(dcd), top=str(prm))
    metal = [
        a.index for a in t.topology.atoms
        if (a.element and a.element.symbol.upper() in METAL_ELEMENTS_LIST)
        or a.name.strip().upper() in METAL_ELEMENTS_LIST
        or a.residue.name.strip().upper() in METAL_ELEMENTS_LIST
    ]
    print("Target metal-like atoms found in trajectory:", len(metal))
else:
    print("prod_dcd missing/too small OR prmtop missing")


In [ ]:
#@title 16) Build solute-complex trajectory for MMPBSA ✅
import mdtraj as md
from pathlib import Path

WD = WORKDIR
solute_dcd = WD / "prod_solute_complex.dcd"
solute_pdb = WD / "prod_solute_complex.pdb"

sel = traj.topology.select("not water and not (element Na or element Cl)")
traj_sol = traj.atom_slice(sel)
traj_sol.save_dcd(str(solute_dcd))
traj_sol[0].save_pdb(str(solute_pdb))
print("Wrote:", solute_dcd.name, solute_pdb.name, "| atoms:", traj_sol.n_atoms, "| frames:", traj_sol.n_frames)


In [ ]:
#@title 16) Build solute-complex trajectory for MMPBSA — safe version ✅
import mdtraj as md
import numpy as np
from pathlib import Path

WD = WORKDIR
solute_dcd = WD / "prod_solute_complex.dcd"
solute_pdb = WD / "prod_solute_complex.pdb"

# Удаляем только реальные solvent/ion residue names.
# НЕ режем по element, чтобы не потерять Cl/Na внутри лиганда/кофактора.
exclude_resnames = {"WAT", "HOH", "TIP3", "TIP3P", "SOL", "NA", "NA+", "CL", "CL-", "SOD", "CLA"}

sel = np.array([
    a.index for a in traj.topology.atoms
    if a.residue.name.strip().upper() not in exclude_resnames
], dtype=int)

traj_sol = traj.atom_slice(sel)

print("Full trajectory atoms:", traj.n_atoms)
print("Solute trajectory atoms:", traj_sol.n_atoms)

traj_sol.save_dcd(str(solute_dcd))
traj_sol[0].save_pdb(str(solute_pdb))

print("Wrote:", solute_dcd.name, solute_pdb.name)

In [ ]:
#@title 17) MM/GBSA via MMPBSA.py (generic ligand selection) ✅
import os, re, textwrap, subprocess
from pathlib import Path
import parmed as pmd

WD = WORKDIR
os.environ["AMBERHOME"] = "/usr/local"

full_prmtop = SYSTEM_PRMTOP
solute_dcd  = WD / "prod_solute_complex.dcd"
assert full_prmtop.exists() and full_prmtop.stat().st_size > 2000
assert solute_dcd.exists() and solute_dcd.stat().st_size > 1_000_000

for fn in [
    "complex_solute.prmtop", "receptor.prmtop", "ligand.prmtop",
    "complex_IFBOX0.prmtop", "receptor_IFBOX0.prmtop", "ligand_IFBOX0.prmtop",
    "FINAL_RESULTS_MMPBSA.dat", "MMPBSA.log", "mmpbsa.in"
]:
    p = WD / fn
    if p.exists():
        p.unlink()

(WD / "_parmed_make_solute.in").write_text(
    "strip :WAT,HOH,TIP3,Na+,Cl-\n"
    "outparm complex_solute.prmtop\n"
    "quit\n"
)
subprocess.run(f"parmed -p {full_prmtop.name} -i _parmed_make_solute.in", shell=True, cwd=str(WD), check=True)

solute_prmtop = WD / "complex_solute.prmtop"
parm = pmd.load_file(str(solute_prmtop))
resnames = [r.name.strip().upper() for r in parm.residues]
nres = len(resnames)

if LIGAND_KIND == "nucleic_acid":
    allowed = {
        "A","C","G","U","DA","DC","DG","DT","RA","RC","RG","RU",
        "DA5","DG5","DT3","DC3","A5","C3","G5",
        MODIFIED_NA_RESNAME.upper()
    }
    lig_res = [i + 1 for i, rn in enumerate(resnames) if rn in allowed]
elif LIGAND_KIND == "small_molecule":
    target = SMALL_MOL_RESNAME.strip().upper()
    lig_res = [i + 1 for i, rn in enumerate(resnames) if rn == target]
else:
    raise ValueError(f"Unknown LIGAND_KIND: {LIGAND_KIND}")

assert len(lig_res) > 0, "Ligand residues were not found in complex_solute.prmtop."
assert lig_res == list(range(min(lig_res), max(lig_res) + 1)), "Ligand residues are not contiguous; adjust ligand selection logic."

rmin, rmax = min(lig_res), max(lig_res)
print("Ligand residue range:", (rmin, rmax), "of", nres)

rec = pmd.load_file(str(solute_prmtop))
rec.strip(f":{rmin}-{rmax}")
rec.save(str(WD / "receptor.prmtop"), overwrite=True)

lig = pmd.load_file(str(solute_prmtop))
if rmax < nres:
    lig.strip(f":{rmax+1}-{nres}")
if rmin > 1:
    lig.strip(f":1-{rmin-1}")
lig.save(str(WD / "ligand.prmtop"), overwrite=True)

def save_ifbox0(src: Path, dst: Path):
    p = pmd.load_file(str(src))
    p.parm_data["POINTERS"][27] = 0
    p.save(str(dst), overwrite=True)

save_ifbox0(solute_prmtop,        WD / "complex_IFBOX0.prmtop")
save_ifbox0(WD / "receptor.prmtop", WD / "receptor_IFBOX0.prmtop")
save_ifbox0(WD / "ligand.prmtop",   WD / "ligand_IFBOX0.prmtop")

def natom(pth: Path) -> int:
    return pmd.load_file(str(pth)).ptr("natom")

n_c = natom(WD / "complex_IFBOX0.prmtop")
n_r = natom(WD / "receptor_IFBOX0.prmtop")
n_l = natom(WD / "ligand_IFBOX0.prmtop")
print("natom:", n_c, n_r, n_l, "sum", n_r + n_l)
assert n_c == n_r + n_l

(WD / "mmpbsa.in").write_text(textwrap.dedent("""
&general
  startframe=1000, endframe=999999, interval=20,
  verbose=1,
  keep_files=0,
  use_sander=1,
/
&gb
  igb=5,
  saltcon=0.150
/
""").strip() + "\n")

cmd = (
    "MMPBSA.py -O "
    "-i mmpbsa.in "
    "-cp complex_IFBOX0.prmtop -rp receptor_IFBOX0.prmtop -lp ligand_IFBOX0.prmtop "
    "-sp complex_IFBOX0.prmtop -y prod_solute_complex.dcd "
    "-o FINAL_RESULTS_MMPBSA.dat"
)
res = subprocess.run(cmd, shell=True, cwd=str(WD), capture_output=True, text=True)
(WD / "MMPBSA.log").write_text(res.stdout + "\n\n=== STDERR ===\n" + res.stderr)

print("Return code:", res.returncode)
print("FINAL_RESULTS exists:", (WD / "FINAL_RESULTS_MMPBSA.dat").exists())
print("Log:", WD / "MMPBSA.log")


In [ ]:
#@title 17) MM/GBSA via MMPBSA.py — Turbo Local & Auto-Stride Version ✅
import os, sys, shutil, subprocess, textwrap
from pathlib import Path
import parmed as pmd
import mdtraj as md

# 1. Настройка путей (Оригиналы на Google Drive, расчет на SSD)
WD_DRIVE = WORKDIR.resolve()
WD_LOCAL = Path("/content/mmpbsa_local_scratch")
WD_LOCAL.mkdir(exist_ok=True, parents=True)

os.environ["AMBERHOME"] = "/usr/local"

# Файлы-источники на Диске
drive_solute_prmtop = WD_DRIVE / "complex_solute.prmtop"
drive_solute_dcd    = WD_DRIVE / "prod_solute_complex.dcd"

# Локальные рабочие файлы на SSD
solute_prmtop = WD_LOCAL / "complex_solute.prmtop"
solute_dcd    = WD_LOCAL / "prod_solute_complex.dcd"
receptor_prmtop = WD_LOCAL / "receptor.prmtop"
ligand_prmtop   = WD_LOCAL / "ligand.prmtop"
mmpbsa_in     = WD_LOCAL / "mmpbsa.in"

final_dat_local = WD_LOCAL / "FINAL_RESULTS_MMPBSA.dat"
mmpbsa_log_local = WD_LOCAL / "MMPBSA.log"

print("--- Старт оптимизированной подготовки MM/GBSA ---")
for p in [drive_solute_prmtop, drive_solute_dcd]:
    assert p.exists(), f"❌ Исходный файл не найден на Google Диске: {p}"

# Быстрое локальное копирование тяжелых файлов на SSD
print("⚡ Кеширование топологии и траектории на локальный SSD...")
shutil.copy(str(drive_solute_prmtop), str(solute_prmtop))
shutil.copy(str(drive_solute_dcd), str(solute_dcd))

# 2. Автоматический расчет оптимального шага (Stride)
print("🧐 Анализ геометрии траектории...")
temp_traj = md.load_topology(str(solute_prmtop))
with md.formats.DCDTrajectoryFile(str(solute_dcd)) as f:
    total_frames = len(f)
print(f"   Всего в траектории найденно кадров: {total_frames}")

# Таргетируем ~200 кадров для идеального баланса скорость/точность
TARGET_FRAMES = 200
auto_interval = max(1, total_frames // TARGET_FRAMES)
print(f"🎯 Профессиональный авто-шаг: interval={auto_interval} (будет обсчитано {total_frames // auto_interval} кадров)")

# 3. Выделение топологий Рецептора и Лиганда силами ParmEd на локальном SSD
print("🧬 Генерация саб-топологий для комплексного ансамбля...")
parm = pmd.load_file(str(solute_prmtop))
resnames = [r.name.strip().upper() for r in parm.residues]
nres = len(resnames)

if LIGAND_KIND == "nucleic_acid":
    allowed = {"A","C","G","U","DA","DC","DG","DT","RA","RC","RG","RU","DA5","DG5","DT3","DC3","A5","C3","G5", MODIFIED_NA_RESNAME.upper()}
    lig_res = [i + 1 for i, rn in enumerate(resnames) if rn in allowed]
elif LIGAND_KIND == "small_molecule":
    target = SMALL_MOL_RESNAME.strip().upper()
    lig_res = [i + 1 for i, rn in enumerate(resnames) if rn == target]
else:
    raise ValueError(f"Unknown LIGAND_KIND: {LIGAND_KIND}")

assert len(lig_res) > 0, "❌ Лиганд не обнаружен в топологии белка!"
rmin, rmax = min(lig_res), max(lig_res)

# Создаем чистый рецептор
rec = pmd.load_file(str(solute_prmtop))
rec.strip(f":{rmin}-{rmax}")
rec.save(str(receptor_prmtop), overwrite=True)

# Создаем чистый лиганд
lig = pmd.load_file(str(solute_prmtop))
if rmax < nres: lig.strip(f":{rmax+1}-{nres}")
if rmin > 1: lig.strip(f":1-{rmin-1}")
lig.save(str(ligand_prmtop), overwrite=True)

# Проверка атомов
n_c = pmd.load_file(str(solute_prmtop)).ptr("natom")
n_r = pmd.load_file(str(receptor_prmtop)).ptr("natom")
n_l = pmd.load_file(str(ligand_prmtop)).ptr("natom")
assert n_c == n_r + n_l, "❌ Ошибка: Сумма атомов рецептора и лиганда не бьется с комплексом!"

# 4. Создание конфигурационного файла MMPBSA.in с авто-шагом
mmpbsa_in.write_text(textwrap.dedent(f"""
&general
  startframe=1, endframe=999999, interval={auto_interval},
  verbose=1,
  keep_files=0,
  use_sander=1,
/
&gb
  igb=5,
  saltcon=0.150
/
""").strip() + "\n")

# 5. Запуск вычислений с ОНЛАЙН СТРИМИНГОМ логов в консоль
cmd = [
    "MMPBSA.py", "-O",
    "-i", str(mmpbsa_in),
    "-cp", str(solute_prmtop),
    "-rp", str(receptor_prmtop),
    "-lp", str(ligand_prmtop),
    "-y", str(solute_dcd),
    "-o", str(final_dat_local),
]

print("\n🚀 Запуск MMPBSA.py ядра на локальном SSD. Стриминг логов выполнения:\n" + "="*60)

# Запускаем процесс и перехватываем поток вывода в реальном времени
process = subprocess.Popen(
    cmd, cwd=str(WD_LOCAL), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

# Выводим строчки по мере их появления в ядре Amber
log_content = []
while True:
    line = process.stdout.readline()
    if not line and process.poll() is not None:
        break
    if line:
        clean_line = line.strip()
        if clean_line:
            print(f"🔥 [MMPBSA]: {clean_line}")
            log_content.append(line)

process.wait()
mmpbsa_log_local.write_text("".join(log_content))

print("="*60 + f"\nВычисления завершены. Код возврата: {process.returncode}")

# 6. Эвакуация результатов обратно на Google Drive
if final_dat_local.exists() and process.returncode == 0:
    print("\n📦 Копирование финальных результатов на Google Диск...")
    shutil.copy(str(final_dat_local), str(WD_DRIVE / "FINAL_RESULTS_MMPBSA.dat"))
    shutil.copy(str(mmpbsa_log_local), str(WD_DRIVE / "MMPBSA.log"))
    print("🟢 ВСЁ ГОТОВО! Файлы сохранены в твою рабочую папку на Диске.")
else:
    print("\n🚨 Ошибка расчета! Сохраняю аварийный лог на Диск для дебага...")
    shutil.copy(str(mmpbsa_log_local), str(WD_DRIVE / "MMPBSA_ERROR.log"))
    raise RuntimeError("MMPBSA.py завершился с ошибкой. Проверь файл MMPBSA_ERROR.log на Диске.")

# Полная очистка тяжелого локального мусора с SSD Colab
shutil.rmtree(WD_LOCAL, ignore_errors=True)

In [ ]:
#@title 18) Parse FINAL_RESULTS_MMPBSA.dat -> publication table ✅
import pandas as pd
import re
from pathlib import Path

WD = WORKDIR
final_dat = WD / "FINAL_RESULTS_MMPBSA.dat"
assert final_dat.exists() and final_dat.stat().st_size > 0

txt = final_dat.read_text(errors="ignore").splitlines()
pat = re.compile(r"^\s*DELTA\s+([A-Z0-9_]+)\s+(-?\d+(?:\.\d+)?(?:[EeDd][-+]?\d+)?)\s+(\d+(?:\.\d+)?(?:[EeDd][-+]?\d+)?)")

rename = {
    "TOTAL": "ΔG_bind (total)",
    "VDWAALS": "ΔE_vdw",
    "EEL": "ΔE_elec",
    "EGB": "ΔG_GB (polar)",
    "ESURF": "ΔG_np (nonpolar)",
    "GGAS": "ΔG_gas",
    "GSOLV": "ΔG_solv",
}
rows = []
for ln in txt:
    m = pat.match(ln.strip())
    if m:
        term = rename.get(m.group(1), m.group(1))
        mean = float(m.group(2).replace("D","E").replace("d","E"))
        sd   = float(m.group(3).replace("D","E").replace("d","E"))
        rows.append([term, mean, sd])

df = pd.DataFrame(rows, columns=["Term", "Mean (kcal/mol)", "SD"]).set_index("Term")
display(df)

out_csv = WD / "MMGBSA_summary_publication.csv"
out_tex = WD / "MMGBSA_summary_publication.tex"
df.to_csv(out_csv)
out_tex.write_text(df.to_latex(escape=False, float_format=lambda x: f"{x:.3f}"))
print("Saved:", out_csv, out_tex)
if "ΔG_bind (total)" in df.index:
    print("ΔG_bind (total):", float(df.loc["ΔG_bind (total)", "Mean (kcal/mol)"]), "kcal/mol")


In [ ]:
from pathlib import Path
import json
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.stats import gaussian_kde
import mdtraj as md

# =========================
# 0) Paths and auto-discovery
# =========================
base_dir = Path(globals().get("WORKDIR", Path.cwd()))
if base_dir.is_file():
    base_dir = base_dir.parent

project_root = Path(globals().get(
    "PROJECT_ROOT",
    base_dir.parents[1] if len(base_dir.parents) >= 2 else base_dir
))

analysis_dir = Path(globals().get("WORKDIR", project_root / "md"))
fig_dir = analysis_dir / "analysis_figures"
fig_dir.mkdir(parents=True, exist_ok=True)

def find_first(patterns, roots):
    roots = [Path(r) for r in roots if Path(r).exists()]
    for root in roots:
        for pat in patterns:
            hits = sorted(root.rglob(pat))
            if hits:
                return hits[0]
    return None

search_roots = [
    analysis_dir,
    project_root,
    project_root.parent,
    Path("/content/drive/MyDrive"),
    Path("/content"),
]

# =========================
# 1) Load trajectory if not already present
# =========================
if "traj" not in globals():
    prmtop_path = globals().get("SYSTEM_PRMTOP", None)
    dcd_path = globals().get("prod_dcd", None)

    if prmtop_path is None or not Path(prmtop_path).exists():
        prmtop_path = find_first(["SYS_*.prmtop", "*.prmtop"], search_roots)
    if dcd_path is None or not Path(dcd_path).exists():
        prod_job = globals().get("PROD_JOB", None)
        patterns = [f"{prod_job}.dcd"] if prod_job else ["*_prod.dcd", "*.dcd"]
        dcd_path = find_first(patterns, search_roots)

    if prmtop_path is None or dcd_path is None:
        raise FileNotFoundError("Could not find prmtop/DCD. Please run the MD cells first or set SYSTEM_PRMTOP / prod_dcd.")

    traj = md.load_dcd(str(dcd_path), top=str(prmtop_path))

# Time axis for production trajectory
prod_dcd_ps = float(globals().get("PROD_DCD_PS", 5.0))
time_ns = np.arange(traj.n_frames) * prod_dcd_ps / 1000.0

# =========================
# 2) Core MD metrics
# =========================
prot_ca = traj.topology.select("protein and name CA")
if len(prot_ca) == 0:
    raise RuntimeError("No protein Cα atoms found in the trajectory topology.")

traj_al = traj.superpose(traj, 0, atom_indices=prot_ca)

rmsd_prot = md.rmsd(traj, traj, 0, atom_indices=prot_ca) * 10.0  # Å
rmsf_prot = md.rmsf(traj_al, traj_al[0], atom_indices=prot_ca) * 10.0  # Å

# Ligand selection
ligand_kind = globals().get("LIGAND_KIND", "small_molecule")
small_mol_resname = str(globals().get("SMALL_MOL_RESNAME", "LIG")).strip().upper()
modified_na_resname = str(globals().get("MODIFIED_NA_RESNAME", "6MA")).strip().upper()

if ligand_kind == "nucleic_acid":
    na_names = {
        "A","C","G","U","DA","DC","DG","DT","RA","RC","RG","RU",
        "DA5","DG5","DT3","DC3","A5","C3","G5",
        modified_na_resname,
    }
    lig_heavy = np.array([
        a.index for a in traj_al.topology.atoms
        if a.residue.name.strip().upper() in na_names and a.element is not None and a.element.symbol != "H"
    ], dtype=int)
elif ligand_kind == "small_molecule":
    lig_heavy = np.array([
        a.index for a in traj_al.topology.atoms
        if a.residue.name.strip().upper() == small_mol_resname and a.element is not None and a.element.symbol != "H"
    ], dtype=int)
else:
    lig_heavy = np.array([], dtype=int)

if len(lig_heavy) > 0:
    rmsd_lig = md.rmsd(traj_al, traj_al[0], atom_indices=lig_heavy) * 10.0  # Å
    lig_com = md.compute_center_of_mass(traj_al.atom_slice(lig_heavy))
    prot_com = md.compute_center_of_mass(traj_al.atom_slice(prot_ca))
    lig_prot_com_dist = np.linalg.norm(lig_com - prot_com, axis=1)  # nm
else:
    rmsd_lig = None
    lig_prot_com_dist = None

# =========================
# 3) Helper functions
# =========================
def save_fig(fig, stem):
    png = fig_dir / f"{stem}.png"
    svg = fig_dir / f"{stem}.svg"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("Saved:", png)
    return png, svg

def parse_mmpbsa_dat(dat_path):
    dat_path = Path(dat_path)
    if not dat_path.exists():
        return None

    txt = dat_path.read_text(errors="ignore").splitlines()
    pat = re.compile(r"^\s*([A-Z ]{3,})\s+(-?\d+(?:\.\d+)?(?:[EeDd][-+]?\d+)?)\s+(\d+(?:\.\d+)?(?:[EeDd][-+]?\d+)?)\s*$")
    block = {}
    for ln in txt:
        m = pat.match(ln.rstrip())
        if not m:
            continue
        key = m.group(1).strip()
        if key in {"VDWAALS", "EEL", "EGB", "ESURF", "G gas", "G solv", "TOTAL"}:
            block[key] = (float(m.group(2).replace("D", "E").replace("d", "E")),
                          float(m.group(3).replace("D", "E").replace("d", "E")))
    return block if block else None

def find_mmpbsa_dat():
    explicit = globals().get("final_dat", None)
    if explicit is not None and Path(explicit).exists():
        return Path(explicit)
    return find_first(["FINAL_RESULTS_MMPBSA*.dat", "FINAL_RESULTS*.dat", "*.dat"], search_roots)

def find_consensus_root():
    candidates = [
        project_root / "consensus_maps",
        project_root / "consensus_maps" / "tables",
        project_root / "consensus_maps" / "json",
    ]
    for c in candidates:
        if c.exists():
            return project_root / "consensus_maps"
    return None

def load_tsv(path):
    return pd.read_csv(path, sep="\t")

# =========================
# 4) 2x2 summary figure
# =========================
fig, axs = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
axs = axs.ravel()

# Panel 1: protein Cα RMSD
axs[0].plot(time_ns, rmsd_prot, lw=1.5)
axs[0].set_title("Protein Cα RMSD")
axs[0].set_xlabel("Time (ns)")
axs[0].set_ylabel("RMSD (Å)")
axs[0].grid(alpha=0.25)

# Panel 2: ligand metric
if rmsd_lig is not None:
    axs[1].plot(time_ns, rmsd_lig, lw=1.5)
    axs[1].set_title("Ligand heavy-atom RMSD")
    axs[1].set_xlabel("Time (ns)")
    axs[1].set_ylabel("RMSD (Å)")
    axs[1].grid(alpha=0.25)
else:
    axs[1].text(0.5, 0.5, "Ligand atoms not found", ha="center", va="center")
    axs[1].set_axis_off()

# Panel 3: protein Cα RMSF
axs[2].plot(np.arange(1, len(rmsf_prot) + 1), rmsf_prot, lw=1.5)
axs[2].set_title("Protein Cα RMSF")
axs[2].set_xlabel("Cα index")
axs[2].set_ylabel("RMSF (Å)")
axs[2].grid(alpha=0.25)

# Panel 4: MMPBSA if present; otherwise ligand-protein COM distance
mmpbsa_dat = find_mmpbsa_dat()
mmpbsa = parse_mmpbsa_dat(mmpbsa_dat) if mmpbsa_dat else None

if mmpbsa and "TOTAL" in mmpbsa:
    keys = [k for k in ["VDWAALS", "EEL", "EGB", "ESURF", "G gas", "G solv", "TOTAL"] if k in mmpbsa]
    vals = [mmpbsa[k][0] for k in keys]
    errs = [mmpbsa[k][1] for k in keys]
    axs[3].bar(keys, vals)
    axs[3].errorbar(keys, vals, yerr=errs, fmt="none", capsize=3)
    axs[3].set_title("MM/GBSA summary")
    axs[3].set_ylabel("kcal/mol")
    axs[3].tick_params(axis="x", rotation=35)
    axs[3].grid(axis="y", alpha=0.25)
else:
    if lig_prot_com_dist is not None:
        axs[3].plot(time_ns, lig_prot_com_dist, lw=1.5)
        axs[3].set_title("Ligand–protein COM distance")
        axs[3].set_xlabel("Time (ns)")
        axs[3].set_ylabel("Distance (nm)")
        axs[3].grid(alpha=0.25)
    else:
        axs[3].text(0.5, 0.5, "No MMPBSA / ligand COM metric", ha="center", va="center")
        axs[3].set_axis_off()

summary_png, summary_svg = save_fig(fig, "md_2x2_summary")
plt.show()

# =========================
# 5) Extra standalone plots
# =========================
# Protein RMSD
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(time_ns, rmsd_prot, lw=1.5)
ax.set_title("Protein Cα RMSD")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("RMSD (Å)")
ax.grid(alpha=0.25)
save_fig(fig, "protein_ca_rmsd")
plt.show()

# Protein RMSD distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(rmsd_prot, bins=30, density=True, alpha=0.5, edgecolor="k")
xg = np.linspace(float(np.min(rmsd_prot)), float(np.max(rmsd_prot)), 400)
ax.plot(xg, gaussian_kde(rmsd_prot)(xg), lw=2)
ax.set_title("Protein Cα RMSD distribution")
ax.set_xlabel("RMSD (Å)")
ax.set_ylabel("Density")
ax.grid(alpha=0.25)
save_fig(fig, "protein_ca_rmsd_distribution")
plt.show()

# Ligand plot if available
if rmsd_lig is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(time_ns, rmsd_lig, lw=1.5)
    ax.set_title("Ligand heavy-atom RMSD")
    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("RMSD (Å)")
    ax.grid(alpha=0.25)
    save_fig(fig, "ligand_rmsd")
    plt.show()

# RMSF
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(rmsf_prot) + 1), rmsf_prot, lw=1.5)
ax.set_title("Protein Cα RMSF")
ax.set_xlabel("Cα index")
ax.set_ylabel("RMSF (Å)")
ax.grid(alpha=0.25)
save_fig(fig, "protein_ca_rmsf")
plt.show()

# =========================
# 6) Jaccard heatmaps and grid maps from docking/consensus outputs
# =========================
consensus_root = find_consensus_root()
summary_rows = []

if consensus_root is not None:
    tables_dir = consensus_root / "tables"
    json_dir = consensus_root / "json"

    # Per-target Jaccard heatmaps
    for jacc_path in sorted(tables_dir.glob("*_jaccard_matrix.tsv")):
        dfj = load_tsv(jacc_path)
        if dfj.empty:
            continue
        mat = dfj.pivot(index="pocket_1", columns="pocket_2", values="jaccard")
        mat = mat.fillna(0.0)

        fig, ax = plt.subplots(figsize=(max(7, 0.45 * mat.shape[1]), max(6, 0.45 * mat.shape[0])))
        im = ax.imshow(mat.values, vmin=0.0, vmax=1.0, cmap="viridis")
        ax.set_xticks(np.arange(mat.shape[1]))
        ax.set_xticklabels(mat.columns, rotation=90, fontsize=8)
        ax.set_yticks(np.arange(mat.shape[0]))
        ax.set_yticklabels(mat.index, fontsize=8)
        ax.set_title(f"Jaccard heatmap — {jacc_path.stem.replace('_jaccard_matrix','')}")
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Jaccard")
        fig.tight_layout()
        save_fig(fig, f"jaccard_{jacc_path.stem.replace('_jaccard_matrix','')}")
        plt.show()

    # Grid overview from consensus global summary
    summary_tsv = tables_dir / "consensus_global_summary.tsv"
    if summary_tsv.exists():
        dfs = load_tsv(summary_tsv)
        if not dfs.empty and {"grid_center_x", "grid_center_y", "grid_size_x", "grid_size_y"}.issubset(dfs.columns):
            fig, ax = plt.subplots(figsize=(8, 7))
            for _, r in dfs.iterrows():
                try:
                    cx = float(r["grid_center_x"]); cy = float(r["grid_center_y"])
                    sx = float(r["grid_size_x"]); sy = float(r["grid_size_y"])
                except Exception:
                    continue

                rect = Rectangle((cx - sx/2.0, cy - sy/2.0), sx, sy, fill=False, lw=1.8)
                ax.add_patch(rect)
                ax.scatter([cx], [cy], s=35)
                ax.text(cx, cy, str(r.get("target_input", "")), fontsize=8, ha="left", va="bottom")

                summary_rows.append({
                    "target_input": r.get("target_input", ""),
                    "grid_center_x": cx, "grid_center_y": cy, "grid_center_z": float(r.get("grid_center_z", np.nan)),
                    "grid_size_x": sx, "grid_size_y": sy, "grid_size_z": float(r.get("grid_size_z", np.nan)),
                    "n_final_consensus_residues": r.get("n_final_consensus_residues", np.nan),
                })

            ax.set_title("Docking grid overview (x–y projection)")
            ax.set_xlabel("Grid center X")
            ax.set_ylabel("Grid center Y")
            ax.grid(alpha=0.25)
            ax.set_aspect("equal", adjustable="datalim")
            save_fig(fig, "grid_overview_xy")
            plt.show()

# =========================
# 7) Save summary table of the main metrics
# =========================
summary = pd.DataFrame([{
    "n_frames": int(traj.n_frames),
    "rmsd_prot_mean_A": float(np.mean(rmsd_prot)),
    "rmsd_prot_max_A": float(np.max(rmsd_prot)),
    "rmsf_prot_mean_A": float(np.mean(rmsf_prot)),
    "rmsf_prot_max_A": float(np.max(rmsf_prot)),
    "ligand_rmsd_mean_A": float(np.mean(rmsd_lig)) if rmsd_lig is not None else np.nan,
    "ligand_rmsd_max_A": float(np.max(rmsd_lig)) if rmsd_lig is not None else np.nan,
    "mmpbsa_total_mean": mmpbsa["TOTAL"][0] if mmpbsa and "TOTAL" in mmpbsa else np.nan,
    "mmpbsa_total_sd": mmpbsa["TOTAL"][1] if mmpbsa and "TOTAL" in mmpbsa else np.nan,
    "traj_time_ns_end": float(time_ns[-1]) if len(time_ns) else np.nan,
}])

summary_csv = fig_dir / "md_summary_metrics.csv"
summary.to_csv(summary_csv, index=False)

if summary_rows:
    grid_summary_csv = fig_dir / "grid_overview_summary.csv"
    pd.DataFrame(summary_rows).to_csv(grid_summary_csv, index=False)
    print("Saved:", grid_summary_csv)

print("Saved:", summary_csv)
print("Done. Figures are in:", fig_dir)

In [ ]:
from pathlib import Path
import json
import math
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm

# --------- robust project discovery ---------
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", "/content/drive/MyDrive/oncotarget_pipeline"))
SEARCH_ROOTS = [
    PROJECT_ROOT,
    PROJECT_ROOT.parent if PROJECT_ROOT.parent != PROJECT_ROOT else PROJECT_ROOT,
    Path.cwd(),
    Path("/mnt/data"),
]

ANALYSIS_DIR = PROJECT_ROOT / "analysis_figures"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

def find_first(pattern: str):
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None

def load_tsv(pattern: str) -> pd.DataFrame:
    path = find_first(pattern)
    if path is None or not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, sep="\t")

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

def save_fig(fig, stem: str):
    png = ANALYSIS_DIR / f"{stem}.png"
    svg = ANALYSIS_DIR / f"{stem}.svg"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print(f"saved: {png}")
    return png, svg

def matrix_from_long(df, row_col, col_col, val_col):
    if df.empty or not {row_col, col_col, val_col}.issubset(df.columns):
        return None
    mat = df.pivot(index=row_col, columns=col_col, values=val_col)
    return mat.sort_index().sort_index(axis=1)

def plot_heatmap(mat: pd.DataFrame, title: str, stem: str, cmap="viridis", vmin=None, vmax=None, center=None, cbar_label=""):
    if mat is None or mat.empty:
        return None
    fig_w = max(7, 0.45 * mat.shape[1] + 3)
    fig_h = max(5, 0.35 * mat.shape[0] + 2.5)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    arr = mat.to_numpy(dtype=float)

    if center is not None:
        norm = TwoSlopeNorm(
            vmin=np.nanmin(arr) if vmin is None else vmin,
            vcenter=center,
            vmax=np.nanmax(arr) if vmax is None else vmax
        )
        im = ax.imshow(arr, aspect="auto", cmap=cmap, norm=norm)
    else:
        im = ax.imshow(arr, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

    ax.set_title(title)
    ax.set_xticks(np.arange(mat.shape[1]))
    ax.set_xticklabels(mat.columns, rotation=90, fontsize=8)
    ax.set_yticks(np.arange(mat.shape[0]))
    ax.set_yticklabels(mat.index, fontsize=8)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if cbar_label:
        cbar.set_label(cbar_label)
    fig.tight_layout()
    save_fig(fig, stem)
    plt.show()
    return fig

# --------- load your generated tables ---------
cons_summary = load_tsv("*consensus_global_summary.tsv")
pocket_tables = sorted([p for root in SEARCH_ROOTS if root.exists() for p in root.rglob("*_pocket_summary.tsv")])
jaccard_tables = sorted([p for root in SEARCH_ROOTS if root.exists() for p in root.rglob("*_jaccard_matrix.tsv")])
votes_tables = sorted([p for root in SEARCH_ROOTS if root.exists() for p in root.rglob("*_residue_votes.tsv")])
aa_tables = sorted([p for root in SEARCH_ROOTS if root.exists() for p in root.rglob("*_aa_composition.tsv")])

best_tsv = find_first("best_per_target.tsv")
md_top3_tsv = find_first("md_candidates_top3_per_target.tsv")
ranked_tsv = find_first("ranked_all_successful.tsv")

best_df = pd.read_csv(best_tsv, sep="\t") if best_tsv else pd.DataFrame()
md3_df = pd.read_csv(md_top3_tsv, sep="\t") if md_top3_tsv else pd.DataFrame()
ranked_df = pd.read_csv(ranked_tsv, sep="\t") if ranked_tsv else pd.DataFrame()

# --------- build one combined table for all targets ---------
combined = cons_summary.copy() if not cons_summary.empty else pd.DataFrame()

if not combined.empty:
    if "methods_used" in combined.columns:
        combined["n_methods_used"] = combined["methods_used"].fillna("").astype(str).apply(
            lambda s: len([x for x in s.split(",") if x.strip()])
        )
    else:
        combined["n_methods_used"] = np.nan

    for c in ["grid_center_x", "grid_center_y", "grid_center_z", "grid_size_x", "grid_size_y", "grid_size_z", "n_final_consensus_residues"]:
        if c not in combined.columns:
            combined[c] = np.nan

    combined["grid_volume_A3"] = safe_num(combined["grid_size_x"]) * safe_num(combined["grid_size_y"]) * safe_num(combined["grid_size_z"])
    combined["grid_diag_A"] = np.sqrt(safe_num(combined["grid_size_x"])**2 + safe_num(combined["grid_size_y"])**2 + safe_num(combined["grid_size_z"])**2)
    combined["consensus_density"] = safe_num(combined["n_final_consensus_residues"]) / combined["grid_volume_A3"].replace(0, np.nan)

    if not ranked_df.empty and "target_input" in ranked_df.columns:
        ranked_agg = ranked_df.groupby("target_input").agg(
            n_successful_docks=("global_job_index", "size"),
            mean_ranked_CNNscore=("CNNscore", "mean"),
            max_ranked_CNNscore=("CNNscore", "max"),
            mean_ranked_minimizedAffinity=("minimizedAffinity", "mean"),
            best_ranked_minimizedAffinity=("minimizedAffinity", "min"),
        ).reset_index()
        combined = combined.merge(ranked_agg, on="target_input", how="left")

    if not best_df.empty and "target_input" in best_df.columns:
        keep = [c for c in ["target_input", "route_type", "CNNscore", "CNNaffinity", "minimizedAffinity"] if c in best_df.columns]
        best_sub = best_df[keep].copy()
        rename_map = {c: f"best_{c}" for c in keep if c != "target_input"}
        best_sub = best_sub.rename(columns=rename_map)
        combined = combined.merge(best_sub, on="target_input", how="left")

    if not md3_df.empty and "target_input" in md3_df.columns:
        md3_agg = md3_df.groupby("target_input").agg(
            md3_mean_CNNscore=("CNNscore", "mean"),
            md3_best_CNNscore=("CNNscore", "max"),
            md3_mean_CNNaffinity=("CNNaffinity", "mean"),
            md3_best_CNNaffinity=("CNNaffinity", "max"),
            md3_mean_minimizedAffinity=("minimizedAffinity", "mean"),
            md3_best_minimizedAffinity=("minimizedAffinity", "min"),
        ).reset_index()
        combined = combined.merge(md3_agg, on="target_input", how="left")

    combined = combined.sort_values(["n_final_consensus_residues", "target_input"], ascending=[False, True])

    combined_path = ANALYSIS_DIR / "combined_target_metrics.tsv"
    combined.to_csv(combined_path, sep="\t", index=False)
    print(f"saved: {combined_path}")

# --------- global target x metric heatmap ---------
if not combined.empty:
    preferred_cols = [
        "n_final_consensus_residues",
        "n_methods_used",
        "grid_center_x", "grid_center_y", "grid_center_z",
        "grid_size_x", "grid_size_y", "grid_size_z",
        "grid_volume_A3", "grid_diag_A",
        "consensus_density",
        "n_successful_docks",
        "mean_ranked_CNNscore", "max_ranked_CNNscore",
        "mean_ranked_minimizedAffinity", "best_ranked_minimizedAffinity",
        "best_CNNscore", "best_CNNaffinity", "best_minimizedAffinity",
        "md3_mean_CNNscore", "md3_best_CNNscore",
        "md3_mean_CNNaffinity", "md3_best_CNNaffinity",
        "md3_mean_minimizedAffinity", "md3_best_minimizedAffinity",
    ]
    metric_cols = [c for c in preferred_cols if c in combined.columns]
    mat = combined.set_index("target_input")[metric_cols].apply(pd.to_numeric, errors="coerce")

    # z-score by column for a clean paper-style overview
    z = (mat - mat.mean(axis=0)) / mat.std(axis=0, ddof=0).replace(0, np.nan)
    z = z.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    plot_heatmap(
        z,
        title="Target × metric overview (z-scores)",
        stem="target_metric_overview_zscore",
        cmap="coolwarm",
        center=0.0,
        cbar_label="z-score",
    )

    # compact raw-value table for archiving
    fig, ax = plt.subplots(figsize=(min(22, 0.8 * len(metric_cols) + 4), max(4, 0.45 * len(combined) + 2)))
    ax.axis("off")
    table_df = combined[["target_input"] + metric_cols].copy()
    tbl = ax.table(
        cellText=table_df.round(3).astype(str).values,
        colLabels=table_df.columns.tolist(),
        loc="center",
        cellLoc="center"
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1, 1.2)
    ax.set_title("Combined target metrics table", pad=20)
    save_fig(fig, "combined_target_metrics_table")
    plt.show()

# --------- per-target pocket-level Jaccard and method-level consensus heatmaps ---------
def plot_target_jaccard_heatmaps(pocket_path: Path, jaccard_path: Path):
    pockets = pd.read_csv(pocket_path, sep="\t")
    jacc = pd.read_csv(jaccard_path, sep="\t")
    target = str(pockets["target_input"].iloc[0]) if "target_input" in pockets.columns and not pockets.empty else pocket_path.stem.replace("_pocket_summary", "")

    # raw pocket-to-pocket matrix
    raw = matrix_from_long(jacc, "pocket_1", "pocket_2", "jaccard")
    if raw is not None and not raw.empty:
        plot_heatmap(raw, f"{target}: pocket-to-pocket Jaccard", f"{target}_pocket_jaccard", cmap="viridis", vmin=0.0, vmax=1.0, cbar_label="Jaccard")

    # method-level matrix = max Jaccard over pocket pairs of the same methods
    if {"method", "pocket_name"}.issubset(pockets.columns) and not jacc.empty:
        pocket_to_method = dict(zip(pockets["pocket_name"].astype(str), pockets["method"].astype(str)))
        methods = sorted(set(pocket_to_method.values()))
        lookup = {}
        for _, r in jacc.iterrows():
            p1 = str(r["pocket_1"]); p2 = str(r["pocket_2"])
            lookup[(p1, p2)] = float(r["jaccard"])

        mm = pd.DataFrame(index=methods, columns=methods, dtype=float)
        for m1 in methods:
            p1s = [p for p, m in pocket_to_method.items() if m == m1]
            for m2 in methods:
                p2s = [p for p, m in pocket_to_method.items() if m == m2]
                vals = []
                for p1 in p1s:
                    for p2 in p2s:
                        v = lookup.get((p1, p2), lookup.get((p2, p1), np.nan))
                        if pd.notna(v):
                            vals.append(float(v))
                mm.loc[m1, m2] = np.nan if not vals else float(np.nanmax(vals))

        plot_heatmap(mm, f"{target}: method consensus (max Jaccard)", f"{target}_method_consensus_jaccard", cmap="viridis", vmin=0.0, vmax=1.0, cbar_label="max Jaccard")

for p in pocket_tables:
    target = p.stem.replace("_pocket_summary", "")
    jp = next((x for x in jaccard_tables if x.stem.replace("_jaccard_matrix", "") == target), None)
    if jp is not None:
        try:
            plot_target_jaccard_heatmaps(p, jp)
        except Exception as e:
            print(f"[skip] {target}: {e}")

# --------- grid map from consensus summary ---------
if not combined.empty and {"grid_center_x", "grid_center_y", "grid_size_x", "grid_size_y"}.issubset(combined.columns):
    fig, ax = plt.subplots(figsize=(8, 7))
    for _, row in combined.iterrows():
        try:
            cx = float(row["grid_center_x"]); cy = float(row["grid_center_y"])
            sx = float(row["grid_size_x"]); sy = float(row["grid_size_y"])
        except Exception:
            continue
        ax.add_patch(Rectangle((cx - sx/2, cy - sy/2), sx, sy, fill=False, lw=1.5))
        ax.scatter([cx], [cy], s=25)
        ax.text(cx, cy, str(row["target_input"]), fontsize=8, ha="left", va="bottom")
    ax.set_title("Docking grid overview (x–y projection)")
    ax.set_xlabel("grid center X")
    ax.set_ylabel("grid center Y")
    ax.grid(alpha=0.25)
    ax.set_aspect("equal", adjustable="datalim")
    save_fig(fig, "grid_overview_xy")
    plt.show()

# --------- optional hooks for external pocket predictors ---------
# Keep these as optional add-ons so they never break the main pipeline.
OPTIONAL_POCKET_TOOLS = {
    "fpocket": {
        "enabled": False,   # switch on only if fpocket is installed and you want to call it here
        "executable": "fpocket",
    },
    "p2rank": {
        "enabled": False,   # switch on only if the standalone P2Rank CLI is installed
        "executable": "p2rank",
    },
}

print("Done. Figures and combined tables are in:", ANALYSIS_DIR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict

# -------------------------------------------------------
# Jaccard similarity
# -------------------------------------------------------
def jaccard_similarity(set_a, set_b):
    a = set(set_a)
    b = set(set_b)
    union = len(a | b)
    if union == 0:
        return 0.0
    return len(a & b) / union


# -------------------------------------------------------
# Build matrix from pockets
# -------------------------------------------------------
def build_jaccard_matrix(pockets_by_algorithm):
    labels = []
    pocket_sets = []
    block_sizes = []

    for alg_name, pocket_list in pockets_by_algorithm.items():
        block_sizes.append(len(pocket_list))
        for i, pocket in enumerate(pocket_list, start=1):
            labels.append(f"{alg_name}\nP{i}")
            pocket_sets.append(set(pocket))

    n = len(pocket_sets)
    matrix = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(n):
            matrix[i, j] = jaccard_similarity(pocket_sets[i], pocket_sets[j])

    return labels, pocket_sets, matrix, block_sizes


# -------------------------------------------------------
# Main heatmap + selection plot
# -------------------------------------------------------
def plot_heatmap_and_selected(pockets_by_algorithm, selected_label="P2Rank\nP1"):
    labels, pocket_sets, matrix, block_sizes = build_jaccard_matrix(pockets_by_algorithm)

    if selected_label not in labels:
        raise ValueError(f"Selected pocket '{selected_label}' not found in labels.")

    selected_idx = labels.index(selected_label)
    selected_set = pocket_sets[selected_idx]

    # Similarity to the selected pocket
    selected_scores = [jaccard_similarity(selected_set, p) for p in pocket_sets]

    # Sort other pockets by similarity for display
    order = np.argsort(selected_scores)[::-1]
    sorted_labels = [labels[i] for i in order]
    sorted_scores = [selected_scores[i] for i in order]
    sorted_colors = ["#d62728" if i == selected_idx else "#1f77b4" for i in order]

    # Figure layout
    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(1, 2, width_ratios=[2.2, 1.0], wspace=0.25)

    # --- Left: heatmap ---
    ax0 = fig.add_subplot(gs[0, 0])
    im = ax0.imshow(matrix, vmin=0.0, vmax=1.0, interpolation="nearest")

    cbar = fig.colorbar(im, ax=ax0, fraction=0.046, pad=0.04)
    cbar.set_label("Сходство Жаккара", rotation=90)

    # Меняем точки на запятые на шкале colorbar
    cbar_ticks = cbar.get_ticks()
    cbar.set_ticklabels([f"{t:.1f}".replace('.', ',') for t in cbar_ticks])

    ax0.set_xticks(np.arange(len(labels)))
    ax0.set_yticks(np.arange(len(labels)))
    ax0.set_xticklabels(labels, fontsize=9)
    ax0.set_yticklabels(labels, fontsize=9)
    plt.setp(ax0.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    ax0.set_title("Попарное сходство Жаккара между предсказанными карманами", fontsize=14, pad=15)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            # Заменяем точку на запятую в ячейках матрицы
            val_str = f"{val:.2f}".replace('.', ',')
            ax0.text(
                j, i, val_str,
                ha="center", va="center",
                fontsize=8,
                color="white" if val < 0.50 else "black"
            )

    cumulative = np.cumsum(block_sizes)
    for b in cumulative[:-1]:
        ax0.axhline(b - 0.5, color="black", linewidth=1.2)
        ax0.axvline(b - 0.5, color="black", linewidth=1.2)

    ax0.set_xlim(-0.5, len(labels) - 0.5)
    ax0.set_ylim(len(labels) - 0.5, -0.5)

    # Highlight selected pocket row/column
    ax0.axhline(selected_idx - 0.5, color="red", linewidth=2.2)
    ax0.axhline(selected_idx + 0.5, color="red", linewidth=2.2)
    ax0.axvline(selected_idx - 0.5, color="red", linewidth=2.2)
    ax0.axvline(selected_idx + 0.5, color="red", linewidth=2.2)

    ax0.text(
        selected_idx, -1.2,
        "Выбранный консенсусный карман",
        ha="center", va="center",
        fontsize=10, color="red", fontweight="bold"
    )

    # --- Right: similarity to selected pocket ---
    ax1 = fig.add_subplot(gs[0, 1])
    y = np.arange(len(sorted_labels))

    bars = ax1.barh(y, sorted_scores, color=sorted_colors, edgecolor="black", linewidth=0.6)
    ax1.set_yticks(y)
    ax1.set_yticklabels(sorted_labels, fontsize=9)
    ax1.invert_yaxis()
    ax1.set_xlim(0, 1.0)

    # Меняем точки на запятые на оси X правого графика
    ax1_ticks = ax1.get_xticks()
    ax1.set_xticks(ax1_ticks)
    ax1.set_xticklabels([f"{t:.1f}".replace('.', ',') for t in ax1_ticks])

    ax1.set_xlabel("Сходство Жаккара")

    # В названии правого графика переносим строку, оставляя имя алгоритма как есть
    selected_label_clean = selected_label.replace('\n', ' ')
    ax1.set_title(f"Сходство с выбранным карманом\n({selected_label_clean})", fontsize=14, pad=12)

    # Highlight the selected pocket itself
    for i, lab in enumerate(sorted_labels):
        if lab == selected_label:
            bars[i].set_color("#d62728")
            bars[i].set_edgecolor("black")
            bars[i].set_linewidth(1.0)

    # Annotate bar values (замена точек на запятые у баров)
    for i, score in enumerate(sorted_scores):
        score_str = f"{score:.2f}".replace('.', ',')
        ax1.text(score + 0.02, i, score_str, va="center", fontsize=9)

    # Threshold line for “consensus”
    consensus_threshold = 0.45
    ax1.axvline(consensus_threshold, color="gray", linestyle="--", linewidth=1.2)

    # Замена точки на запятую в тексте порога
    threshold_str = f"{consensus_threshold:.2f}".replace('.', ',')
    ax1.text(consensus_threshold + 0.01, len(sorted_labels) - 0.5,
             f"порог = {threshold_str}",
             fontsize=9, color="gray", va="bottom")

    fig.suptitle("Выбор консенсусного кармана по попарному сходству Жаккара", fontsize=16, y=0.98)
    fig.tight_layout()
    return fig, (ax0, ax1)


# -------------------------------------------------------
# Demo synthetic pockets with a consensus pocket
# -------------------------------------------------------
def make_variant_pocket(core, universe, keep_frac=0.8, add_frac=0.12, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    core = np.array(sorted(core))
    outside = np.setdiff1d(universe, core)

    keep_n = max(4, int(round(len(core) * keep_frac)))
    keep_n = min(keep_n, len(core))
    kept = set(rng.choice(core, size=keep_n, replace=False))

    add_n = max(2, int(round(len(core) * add_frac)))
    add_n = min(add_n, len(outside))
    added = set(rng.choice(outside, size=add_n, replace=False))

    return kept | added


def build_consensus_demo_pockets(seed=11):
    rng = np.random.default_rng(seed)
    universe = np.arange(1, 401)

    consensus_core = set(rng.choice(np.arange(120, 170), size=28, replace=False))
    alt_core_1 = set(rng.choice(np.arange(20, 70), size=22, replace=False))
    alt_core_2 = set(rng.choice(np.arange(210, 260), size=24, replace=False))
    alt_core_3 = set(rng.choice(np.arange(300, 360), size=23, replace=False))

    pockets_by_algorithm = OrderedDict()

    pockets_by_algorithm["P2Rank"] = [
        make_variant_pocket(consensus_core, universe, keep_frac=0.88, add_frac=0.08, rng=rng),  # P1
        make_variant_pocket(alt_core_1, universe, keep_frac=0.82, add_frac=0.12, rng=rng),      # P2
        make_variant_pocket(alt_core_3, universe, keep_frac=0.80, add_frac=0.12, rng=rng),      # P3
    ]

    pockets_by_algorithm["fpocket"] = [
        make_variant_pocket(alt_core_2, universe, keep_frac=0.78, add_frac=0.14, rng=rng),      # P1
        make_variant_pocket(consensus_core, universe, keep_frac=0.84, add_frac=0.10, rng=rng),  # P2
        make_variant_pocket(alt_core_1, universe, keep_frac=0.77, add_frac=0.15, rng=rng),      # P3
    ]

    pockets_by_algorithm["COACH-D"] = [
        make_variant_pocket(consensus_core, universe, keep_frac=0.90, add_frac=0.07, rng=rng),  # P1
        make_variant_pocket(alt_core_2, universe, keep_frac=0.80, add_frac=0.12, rng=rng),      # P2
        make_variant_pocket(alt_core_3, universe, keep_frac=0.76, add_frac=0.14, rng=rng),      # P3
    ]

    pockets_by_algorithm["DoGSiteScorer"] = [
        make_variant_pocket(alt_core_1, universe, keep_frac=0.76, add_frac=0.14, rng=rng),      # P1
        make_variant_pocket(alt_core_2, universe, keep_frac=0.78, add_frac=0.13, rng=rng),      # P2
        make_variant_pocket(consensus_core, universe, keep_frac=0.85, add_frac=0.12, rng=rng),   # P3
    ]

    return pockets_by_algorithm


# -------------------------------------------------------
# Run
# -------------------------------------------------------
demo_pockets = build_consensus_demo_pockets(seed=11)
fig, axes = plot_heatmap_and_selected(
    demo_pockets,
    selected_label="P2Rank\nP1"
)
plt.show()